In [10]:
import akshare as ak
ak.index_stock_cons_weight_csindex(symbol="000300")
ak.index_csindex_all()

import akshare as ak
import pandas as pd

# 获取数据
df = ak.currency_boc_safe()

# 筛选港币数据
hkd_mid = df[['日期', '港元']].copy()
hkd_mid.columns = ['date', 'hkd_cny_mid']

# 转换为标准格式
hkd_mid['hkd_cny_per_1'] = hkd_mid['hkd_cny_mid'] / 100

# 【修复】将 date 列转换为字符串
hkd_mid['date'] = hkd_mid['date'].astype(str)

# 筛选数据
hkd_mid = hkd_mid[hkd_mid['date'] >= '1990-12-19'].reset_index(drop=True)

print(f"数据范围：{hkd_mid['date'].min()} ~ {hkd_mid['date'].max()}")
print(hkd_mid)

from forex_python.converter import CurrencyRates
from datetime import datetime

c = CurrencyRates()
date = datetime(1994, 1, 1)
historical_rates = c.get_rates('USD', date)
print("2022年1月1日USD对其他货币的汇率:")
for currency, rate in historical_rates.items():
    print(f"{currency}: {rate}")

数据范围：1994-01-01 ~ 2026-02-27
            date  hkd_cny_mid  hkd_cny_per_1
0     1994-01-01      112.660        1.12660
1     1994-01-03      112.660        1.12660
2     1994-01-04      112.660        1.12660
3     1994-01-05      112.660        1.12660
4     1994-01-06      112.660        1.12660
...          ...          ...            ...
7907  2026-02-13       88.787        0.88787
7908  2026-02-24       88.773        0.88773
7909  2026-02-25       88.611        0.88611
7910  2026-02-26       88.557        0.88557
7911  2026-02-27       88.484        0.88484

[7912 rows x 3 columns]


RatesNotAvailableError: Currency Rates Source Not Ready

In [19]:
class ExchangeRateClient:
    """统一汇率客户端，支持多数据源"""
    
    def __init__(self, source: str = "frankfurter", fred_api_key: str = None):
        self.source = source
        self.fred_api_key = fred_api_key
        self.base_urls = {
            "frankfurter": "https://api.frankfurter.app",
            "exchangerate": "https://api.exchangerate.host",
            "fred": "https://api.stlouisfed.org/fred/series/observations",
        }
    
    def get_usd_hkd(self, date: str = "latest") -> float:
        """获取 USD/HKD 汇率"""
        
        if self.source == "frankfurter":
            return self._get_frankfurter(date)
        elif self.source == "fred":
            return self._get_fred(date)
        else:
            raise ValueError(f"Unknown source: {self.source}")
    
    def _get_frankfurter(self, date: str) -> float:
        import requests
        url = f"{self.base_urls['frankfurter']}/{date}"
        params = {"from": "USD", "to": "HKD"}
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        return resp.json()["rates"]["HKD"]
    
    def _get_fred(self, date: str) -> float:
        import requests
        if self.fred_api_key is None:
            raise ValueError("FRED API Key required")
        
        params = {
            "series_id": "DEXHKUS",
            "api_key": self.fred_api_key,
            "file_type": "json",
            "observation_start": date,
            "observation_end": date,
        }
        resp = requests.get(self.base_urls["fred"], params=params)
        resp.raise_for_status()
        data = resp.json()
        obs = data["observations"]
        if not obs or obs[0]["value"] == ".":
            raise ValueError("No data for date")
        return float(obs[0]["value"])


# 使用
client = ExchangeRateClient(source="frankfurter")
rate = client.get_usd_hkd("1994-05-22")
print(f"1 USD = {rate:.4f} HKD")

HTTPError: 404 Client Error: Not Found for url: https://api.frankfurter.app/1994-05-22?from=USD&to=HKD

In [20]:
import requests
import json
import os
from datetime import datetime

class FrankfurterCache:
    """Frankfurter 汇率全量缓存类"""
    
    def __init__(self, cache_file: str = "frankfurter_usd_hkd.json"):
        self.cache_file = cache_file
        self.data = self._load_cache()
    
    def _load_cache(self) -> dict:
        """加载缓存文件"""
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except:
                pass
        return {}
    
    def _save_cache(self):
        """保存缓存到文件"""
        with open(self.cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.data, f, ensure_ascii=False, indent=2)
    
    def fetch_all_history(self, start_date: str = "1999-01-04", end_date: str = None):
        """
        一次性获取全部历史数据并缓存
        :param start_date: 起始日期 (Frankfurter 最早 1999-01-04)
        :param end_date: 结束日期，默认今天
        """
        if end_date is None:
            end_date = datetime.now().strftime('%Y-%m-%d')
        
        print(f"🔄 正在获取 {start_date} 至 {end_date} 的汇率数据...")
        
        # 批量请求日期范围
        url = f"https://api.frankfurter.app/{start_date}..{end_date}"
        params = {"from": "USD", "to": "HKD"}
        
        resp = requests.get(url, params=params, timeout=120)  # 大数据量需延长超时
        resp.raise_for_status()
        data = resp.json()
        
        # 解析返回数据
        # 返回格式：{"amount":1,"base":"USD","start_date":"...","end_date":"...","rates":{"YYYY-MM-DD":{"HKD":7.80}}}
        rates = data.get('rates', {})
        
        # 转换为扁平字典：{"YYYY-MM-DD": 7.80}
        self.data = {
            date: float(rates[date]["HKD"])
            for date in rates
            if "HKD" in rates[date] and rates[date]["HKD"] is not None
        }
        
        # 保存缓存
        self._save_cache()
        
        print(f"✅ 获取成功：{len(self.data)} 条记录")
        print(f"📊 数据范围：{min(self.data.keys())} ~ {max(self.data.keys())}")
        print(f"💾 缓存文件：{self.cache_file}")
        
        return self.data
    
    def get_rate(self, date: str) -> float:
        """
        获取指定日期的汇率（自动从缓存读取）
        :param date: "YYYY-MM-DD"
        :return: 1 USD = XXX HKD
        """
        # 标准化日期格式
        date = date.strip()
        
        # 直接命中
        if date in self.data:
            return self.data[date]
        
        # 向前查找最近的交易日（处理周末/节假日）
        from datetime import timedelta
        d = datetime.strptime(date, '%Y-%m-%d')
        for i in range(1, 10):  # 最多向前找 10 天
            prev_date = (d - timedelta(days=i)).strftime('%Y-%m-%d')
            if prev_date in self.data:
                return self.data[prev_date]
        
        # 兜底：返回缓存中的最新汇率
        if self.data:
            return self.data[max(self.data.keys())]
        
        raise KeyError(f"未找到 {date} 及附近的汇率数据")
    
    def get_rates_batch(self, dates: list) -> dict:
        """批量获取多个日期的汇率"""
        return {date: self.get_rate(date) for date in dates}
    
    def update_cache(self):
        """增量更新缓存（获取最新数据）"""
        if not self.data:
            return self.fetch_all_history()
        
        # 获取缓存中最新日期
        latest_date = max(self.data.keys())
        today = datetime.now().strftime('%Y-%m-%d')
        
        if latest_date >= today:
            print("✅ 缓存已是最新")
            return
        
        # 只获取缺失的部分
        print(f"🔄 增量更新：{latest_date} 至 {today}")
        url = f"https://api.frankfurter.app/{latest_date}..{today}"
        params = {"from": "USD", "to": "HKD"}
        
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        
        rates = data.get('rates', {})
        for date in rates:
            if "HKD" in rates[date] and rates[date]["HKD"]:
                self.data[date] = float(rates[date]["HKD"])
        
        self._save_cache()
        print(f"✅ 更新完成：共 {len(self.data)} 条记录")


# ============================================
# 使用示例
# ============================================
if __name__ == "__main__":
    cache = FrankfurterCache("frankfurter_usd_hkd.json")
    
    # 1. 首次运行：获取全部历史数据
    if len(cache.data) == 0:
        cache.fetch_all_history("1999-01-04")
    
    # 2. 日常运行：增量更新
    else:
        cache.update_cache()
    
    # 3. 查询汇率
    rate = cache.get_rate("2020-05-20")
    print(f"\n2020-05-20: 1 USD = {rate:.4f} HKD")
    
    # 4. 批量查询（港股分红复权场景）
    ex_dates = ["2020-05-20", "2019-06-12", "2015-03-05", "2010-01-15"]
    rates = cache.get_rates_batch(ex_dates)
    print("\n批量查询结果:")
    for date, rate in rates.items():
        print(f"  {date}: {rate:.4f}")

🔄 正在获取 1999-01-04 至 2026-03-01 的汇率数据...
✅ 获取成功：1417 条记录
📊 数据范围：1999-01-04 ~ 2026-02-23
💾 缓存文件：frankfurter_usd_hkd.json

2020-05-20: 1 USD = 7.7524 HKD

批量查询结果:
  2020-05-20: 7.7524
  2019-06-12: 7.8310
  2015-03-05: 7.7558
  2010-01-15: 7.7565


In [23]:
import requests
import csv
import os
import bisect
from datetime import datetime

class FrankfurterCache:
    """汇率缓存 - 二分查找极简版"""
    
    def __init__(self, cache_file: str = "frankfurter_usd_hkd.csv"):
        self.cache_file = cache_file
        self.data = {}           # {"YYYY-MM-DD": rate}
        self.sorted_dates = []   # 已排序的日期列表
        self._load_cache()
    
    def _load_cache(self):
        """加载 CSV 缓存"""
        if not os.path.exists(self.cache_file):
            return
        with open(self.cache_file, 'r', encoding='utf-8', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.data[row['date']] = float(row['usd_hkd'])
        self.sorted_dates = sorted(self.data.keys())
    
    def _save_cache(self):
        """保存 CSV 缓存（按日期排序）"""
        with open(self.cache_file, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['date', 'usd_hkd'])
            for date in self.sorted_dates:
                writer.writerow([date, f"{self.data[date]:.6f}"])
    
    def get_rate(self, date: str) -> float:
        """
        获取汇率：二分查找 ≤ target_date 的最大日期
        时间复杂度：O(log n)
        """
        if not self.sorted_dates:
            raise RuntimeError("缓存为空，请先调用 fetch_all_history()")
        
        date = date.strip()
        
        # 二分查找：找到插入位置
        idx = bisect.bisect_right(self.sorted_dates, date) - 1
        
        # 边界：早于最早数据 → 返回最早日期汇率
        if idx < 0:
            return self.data[self.sorted_dates[0]]
        
        # 返回找到的日期汇率
        return self.data[self.sorted_dates[idx]]
    
    def get_rates_batch(self, dates: list) -> dict:
        """批量获取汇率"""
        return {date: self.get_rate(date) for date in dates}
    
    def fetch_all_history(self, start_date: str = "1999-01-04", end_date: str = None):
        """获取全部历史数据"""
        if end_date is None:
            end_date = datetime.now().strftime('%Y-%m-%d')
        
        print(f"🔄 获取 {start_date} 至 {end_date}...")
        url = f"https://api.frankfurter.app/{start_date}..{end_date}"
        params = {"from": "USD", "to": "HKD"}
        
        resp = requests.get(url, params=params, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        
        rates = data.get('rates', {})
        self.data = {
            date: float(rates[date]["HKD"])
            for date in rates
            if "HKD" in rates[date] and rates[date]["HKD"]
        }
        self.sorted_dates = sorted(self.data.keys())
        self._save_cache()
        
        print(f"✅ {len(self.data)} 条记录 | {self.sorted_dates[0]} ~ {self.sorted_dates[-1]}")
    
    def update_cache(self):
        """增量更新"""
        if not self.sorted_dates:
            return self.fetch_all_history()
        
        latest = self.sorted_dates[-1]
        today = datetime.now().strftime('%Y-%m-%d')
        if latest >= today:
            return
        
        print(f"🔄 更新 {latest} 至 {today}")
        url = f"https://api.frankfurter.app/{latest}..{today}"
        resp = requests.get(url, params={"from": "USD", "to": "HKD"}, timeout=60)
        data = resp.json().get('rates', {})
        
        for date in data:
            if "HKD" in data[date] and data[date]["HKD"]:
                self.data[date] = float(data[date]["HKD"])
        
        self.sorted_dates = sorted(self.data.keys())
        self._save_cache()
        print(f"✅ 共 {len(self.data)} 条记录")


# ============================================
# 使用示例
# ============================================
if __name__ == "__main__":
    cache = FrankfurterCache("frankfurter_usd_hkd.csv")
    
    # 首次运行获取全量数据
    if not cache.sorted_dates:
        cache.fetch_all_history()
    else:
        cache.update_cache()
    
    # 查询测试
    test_dates = [
        "2026-01-25",  # 周日 → 用周五
        "2026-01-20",  # 节假日 → 用前一交易日
        "2026-02-17",  # 交易日 → 精确匹配
        "1999-01-01",  # 早于最早 → 用最早
        "2099-12-31",  # 晚于最新 → 用最新
    ]
    
    print("\n汇率查询测试:")
    for date in test_dates:
        rate = cache.get_rate(date)
        print(f"  {date} → {rate:.4f}")
    
    # 港股分红复权
    dividends = [
        {"symbol": "00700.HK", "ex_date": "2026-01-25", "dividend_usd": 0.50},
        {"symbol": "00941.HK", "ex_date": "2026-02-14", "dividend_usd": 1.20},
    ]
    
    print("\n分红复权计算:")
    for item in dividends:
        rate = cache.get_rate(item["ex_date"])
        hkd = item["dividend_usd"] * rate
        print(f"{item['symbol']} | {item['ex_date']} | {item['dividend_usd']:.2f} USD × {rate:.4f} = {hkd:.4f} HKD")

🔄 更新 2026-02-27 至 2026-03-01
✅ 共 1421 条记录

汇率查询测试:
  2026-01-25 → 7.7977
  2026-01-20 → 7.7977
  2026-02-17 → 7.8152
  1999-01-01 → 7.7474
  2099-12-31 → 7.8237

分红复权计算:
00700.HK | 2026-01-25 | 0.50 USD × 7.7977 = 3.8988 HKD
00941.HK | 2026-02-14 | 1.20 USD × 7.8166 = 9.3799 HKD
